# Treinamento v4.0 (Voz Real do Luiz + Marcus Braga Pareado) — `luiz-academico`
**O que mudou na v4.0:**
1. **Foco Principal nos 3 Textos 95–100% Humanos do Luiz:** 3 textos completos (4x) + 8 blocos alinhados (4x) + 20 frases manuais (2x) + 20 escolhas de próximo token (2x) = 124 linhas.
2. **Alicerce 100% Humano Pré-LLM:** 273 blocos do Marcus Braga pareados com entradas de IA geradas pelo DeepSeek (zero mistura com textos contaminados antigos).
3. **Treino Direto & Exportação Segura:** Sem geração prévia na GPU — lê `dataset_treino_v4_0.jsonl` direto (397 pares) e exporta GGUF Q4_K_M.

In [ ]:
# CÉLULA 1: Conectar Google Drive e garantir dataset_treino_v4_0.jsonl
import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')

drive_ds = "/content/drive/MyDrive/dataset_treino_v4_0.jsonl"
local_ds = "dataset_treino_v4_0.jsonl"

if not os.path.exists(local_ds):
    if os.path.exists(drive_ds):
        shutil.copy(drive_ds, local_ds)
    else:
        print("Selecione o arquivo 'dataset_treino_v4_0.jsonl' do seu computador:")
        uploaded = files.upload()
        up_name = list(uploaded.keys())[0]
        if up_name != local_ds:
            shutil.copy(up_name, local_ds)

print(f"✅ {local_ds} pronto com {sum(1 for _ in open(local_ds, encoding='utf-8'))} pares!")

In [ ]:
# CÉLULA 2: Instalar Unsloth e dependências
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# CÉLULA 3: Carregar Llama 3.1 8B Base e treinar SFT (397 pares, QLoRA r=32, alpha=64, ResponseOnlyCollator)
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling
import gc, torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

alpaca_prompt = """Abaixo está uma instrução que descreve uma tarefa, acompanhada de uma entrada que fornece o contexto. Escreva uma resposta que conclua adequadamente a solicitação sem qualquer metatexto, mantendo exatamente o mesmo número de parágrafos e todos os fatos da entrada.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    texts = [
        alpaca_prompt.format(inst, inp, out) + EOS_TOKEN
        for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"])
    ]
    return {"text": texts}

sft_dataset = load_dataset("json", data_files="dataset_treino_v4_0.jsonl", split="train")
sft_dataset = sft_dataset.map(formatting_prompts_func, batched=True)

class ResponseOnlyCollator(DataCollatorForLanguageModeling):
    def __init__(self, tokenizer):
        super().__init__(tokenizer=tokenizer, mlm=False)
        self.resp_ids = tokenizer.encode("### Resposta:\n", add_special_tokens=False)
    def torch_call(self, examples):
        batch = super().torch_call(examples)
        labels = batch["labels"]
        m = len(self.resp_ids)
        for i in range(labels.size(0)):
            seq = labels[i].tolist()
            for j in range(len(seq) - m + 1):
                if seq[j:j+m] == self.resp_ids:
                    labels[i, :j+m] = -100
                    break
        return batch

pasta_v4 = "/content/lora_luiz_v4_0_final"
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = ResponseOnlyCollator(tokenizer),
    dataset_num_proc = 2,
    packing = False,
    neftune_noise_alpha = 5,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 12,
        num_train_epochs = 3,
        learning_rate = 1.5e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        save_strategy = "no",
        optim = "adamw_8bit",
        weight_decay = 0.03,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = pasta_v4,
        report_to = "none",
    ),
)

trainer.train()
model.save_pretrained(pasta_v4)
tokenizer.save_pretrained(pasta_v4)
del trainer, sft_dataset
gc.collect(); torch.cuda.empty_cache()
print(f"✅ Treino v4.0 salvo em: {pasta_v4}")

In [ ]:
# CÉLULA 4: Exportar GGUF (Q4_K_M) e copiar para o Drive (se houver cota) ou baixar direto
import glob, shutil, os, gc, torch

gc.collect(); torch.cuda.empty_cache()
!rm -rf /content/tmp_gguf*

model.save_pretrained_gguf(
    "/content/tmp_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
    maximum_memory_usage = 0.35,
)

ggufs = glob.glob("/content/tmp_gguf*/*Q4_K_M.gguf")
if ggufs:
    src = ggufs[0]
    dst = "/content/drive/MyDrive/luiz-academico-v4_0-q4_k_m.gguf"
    try:
        shutil.copy(src, dst)
        print(f"🎉 SUCESSO TOTAL! Modelo v4.0 (4.9 GB) salvo no Drive em: {dst}")
    except Exception as e:
        print(f"⚠️ Não copiou para o Drive ({e}). Baixe direto pelo painel esquerdo em: {src}")
else:
    print("⚠️ Arquivo não encontrado em /content/tmp_gguf*. Conteúdo:", glob.glob("/content/tmp_gguf*/*"))